# 🔄 Olist Customer Churn Prediction

**Objective:** Predict whether a customer will churn (no purchase within a defined window) using behavioral features extracted from the Olist DuckDB Data Warehouse.

**Models used:**
- Gradient Boosting Machine (GBM)
- Random Forest
- Logistic Regression

**Evaluation metrics:** Accuracy, AUC-ROC, Precision, Recall

---

## 1. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import duckdb
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
)

# ── Config ──────────────────────────────────────────────────────────────────
DB_PATH        = "olist_warehouse.duckdb"
CHURN_THRESHOLD = 180   # days — customer is 'churned' if no order within this window
TEST_SIZE       = 0.20
RANDOM_STATE    = 42

print(f"DuckDB version : {duckdb.__version__}")
print(f"Churn threshold: >{CHURN_THRESHOLD} days inactive = churned")

## 2. Connect to DuckDB Data Warehouse

In [ ]:
con = duckdb.connect(DB_PATH, read_only=False)

# Inspect available tables
tables = con.execute("SHOW TABLES").df()
print("Tables in warehouse:")
for t in tables["name"]:
    cnt = con.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]
    print(f"  • {t}: {cnt:,} rows")

## 3. Feature Engineering — Customer-Level Aggregation

We join `Fact_Sales` with `Dim_Customer` and compute the following features per customer:

| Feature | Description |
|---|---|
| `total_orders` | Number of distinct orders |
| `total_spent` | Total revenue from customer |
| `avg_order_value` | Mean order price |
| `avg_freight_ratio` | Mean freight-to-price ratio |
| `avg_review_score` | Mean satisfaction score (1–5) |
| `avg_delivery_days` | Mean delivery time |
| `unique_categories` | Breadth of product exploration |
| `unique_sellers` | Number of distinct sellers used |
| `customer_lifespan_days` | Days between first and last order |
| `days_since_last_order` | Recency — used to derive the churn label |
| `customer_state_enc` | One-hot encoded top-5 states |

In [ ]:
df_raw = con.execute("""
    SELECT
        f.customer_id,
        COUNT(DISTINCT f.order_id)                         AS total_orders,
        SUM(f.price)                                       AS total_spent,
        AVG(f.price)                                       AS avg_order_value,
        AVG(f.freight_value / NULLIF(f.price, 0))          AS avg_freight_ratio,
        AVG(CAST(f.review_score AS DOUBLE))                AS avg_review_score,
        AVG(CAST(f."Delivery Days" AS DOUBLE))             AS avg_delivery_days,
        COUNT(DISTINCT f.product_category_name)            AS unique_categories,
        COUNT(DISTINCT f.seller_id)                        AS unique_sellers,
        MIN(CAST(f.order_purchase_timestamp AS DATE))      AS first_order_date,
        MAX(CAST(f.order_purchase_timestamp AS DATE))      AS last_order_date,
        DATEDIFF('day',
            MIN(CAST(f.order_purchase_timestamp AS DATE)),
            MAX(CAST(f.order_purchase_timestamp AS DATE))) AS customer_lifespan_days,
        DATEDIFF('day',
            MAX(CAST(f.order_purchase_timestamp AS DATE)),
            (SELECT MAX(CAST(order_purchase_timestamp AS DATE))
             FROM "Fact_Sales"))                           AS days_since_last_order,
        c.customer_state
    FROM "Fact_Sales" f
    LEFT JOIN "Dim_Customer" c ON f.customer_id = c.customer_id
    GROUP BY f.customer_id, c.customer_state
""").df()

print(f"Raw customer rows: {len(df_raw):,}")
df_raw.head()

## 4. Label Creation & Preprocessing

In [ ]:
df = df_raw.copy()

# ── Churn label ──────────────────────────────────────────────────────────────
df["churned"] = (df["days_since_last_order"] > CHURN_THRESHOLD).astype(int)

churn_rate = df["churned"].mean() * 100
print(f"Churn rate : {churn_rate:.1f}%")
print(df["churned"].value_counts().rename({0: "Retained", 1: "Churned"}))

# ── Encode top-5 states ───────────────────────────────────────────────────────
top_states = df["customer_state"].value_counts().head(5).index
df["customer_state_enc"] = df["customer_state"].apply(
    lambda x: x if x in top_states else "Other"
)
df = pd.get_dummies(df, columns=["customer_state_enc"], drop_first=True)

# ── Drop non-feature columns ──────────────────────────────────────────────────
df = df.drop(columns=["customer_id", "customer_state",
                       "first_order_date", "last_order_date"])

# ── Impute remaining NaNs with median ────────────────────────────────────────
df = df.fillna(df.median(numeric_only=True))

print(f"\nFinal feature matrix: {df.shape}")
df.head()

## 5. Train / Test Split

In [ ]:
# Remove target AND leakage column (days_since_last_order directly defines churn)
X = df.drop(columns=["churned", "days_since_last_order"])
y = df["churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set : {X_train.shape[0]:,} rows")
print(f"Test set     : {X_test.shape[0]:,} rows")
print(f"Features     : {X_train.shape[1]}")
print(f"Feature names: {list(X_train.columns)}")

## 6. Model Training

Three classifiers are trained and evaluated:

1. **GBM** — 200 trees, shallow depth, low learning rate (robust, slow overfitting)
2. **Random Forest** — 200 trees, balanced class weights (handles imbalance)
3. **Logistic Regression** — with StandardScaler (interpretable baseline)

In [ ]:
ch_models = {
    "GBM": GradientBoostingClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.05, random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=500, class_weight="balanced")),
    ]),
}

results, preds, probas = {}, {}, {}

for name, mdl in ch_models.items():
    print(f"Training {name}...", end=" ")
    mdl.fit(X_train, y_train)
    yp  = mdl.predict(X_test)
    ypr = mdl.predict_proba(X_test)[:, 1]
    rep = classification_report(y_test, yp, output_dict=True)
    results[name] = {
        "Accuracy":  round((yp == y_test).mean() * 100, 1),
        "AUC-ROC":   round(roc_auc_score(y_test, ypr), 3),
        "Precision": round(rep["1"]["precision"], 3),
        "Recall":    round(rep["1"]["recall"], 3),
    }
    preds[name], probas[name] = yp, ypr
    print(f"AUC={results[name]['AUC-ROC']}")

best_name = max(results, key=lambda k: results[k]["AUC-ROC"])
best_mdl  = ch_models[best_name]
print(f"\n🏆 Best model: {best_name} (AUC-ROC = {results[best_name]['AUC-ROC']})")

## 7. Results Summary

In [ ]:
res_df = pd.DataFrame(results).T
res_df.index.name = "Model"
res_df = res_df.reset_index()
print(res_df.to_string(index=False))

## 8. Classification Report — Best Model

In [ ]:
print(f"Classification Report — {best_name}")
print("=" * 50)
print(classification_report(
    y_test, preds[best_name],
    target_names=["Retained (0)", "Churned (1)"]
))

## 9. ROC Curve — All Models

In [ ]:
colors = {
    "GBM": "#4C8EDA",
    "Random Forest": "#2ecc71",
    "Logistic Regression": "#FF6B35"
}

fig_roc = go.Figure()
for name in results:
    fpr, tpr, _ = roc_curve(y_test, probas[name])
    fig_roc.add_trace(go.Scatter(
        x=fpr, y=tpr,
        name=f"{name} (AUC={results[name]['AUC-ROC']})",
        line=dict(color=colors[name], width=2.5),
    ))

# Diagonal baseline
fig_roc.add_shape(
    type="line", x0=0, y0=0, x1=1, y1=1,
    line=dict(dash="dash", color="#7c8db0", width=1)
)

fig_roc.update_layout(
    title="ROC Curve — All Models",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    template="plotly_dark",
    legend=dict(orientation="h", y=-0.22),
    width=800, height=450,
)
fig_roc.show()

## 10. Confusion Matrix — Best Model

In [ ]:
cm = confusion_matrix(y_test, preds[best_name])

fig_cm = ff.create_annotated_heatmap(
    z=cm,
    x=["Pred: Stay", "Pred: Churn"],
    y=["Actual: Stay", "Actual: Churn"],
    colorscale=[[0, "#1e2130"], [1, "#4C8EDA"]],
    showscale=False,
)
fig_cm.update_layout(
    title=f"Confusion Matrix — {best_name}",
    template="plotly_dark",
    width=500, height=400,
)
fig_cm.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives  (Stay→Stay)    : {tn:,}")
print(f"False Positives (Stay→Churn)   : {fp:,}")
print(f"False Negatives (Churn→Stay)   : {fn:,}")
print(f"True Positives  (Churn→Churn)  : {tp:,}")

## 11. Feature Importance — Best Model

In [ ]:
estimator = (
    best_mdl.named_steps["model"]
    if hasattr(best_mdl, "named_steps")
    else best_mdl
)

imp_vals = (
    estimator.feature_importances_
    if hasattr(estimator, "feature_importances_")
    else np.abs(estimator.coef_[0])
)

importances = (
    pd.Series(imp_vals, index=X_train.columns)
    .sort_values(ascending=False)
    .head(10)
)

fig_imp = go.Figure(go.Bar(
    x=importances.values[::-1],
    y=importances.index[::-1],
    orientation="h",
    marker=dict(
        color=list(range(len(importances))),
        colorscale=[[0, "#2ecc71"], [1, "#4C8EDA"]],
        showscale=False,
    ),
    text=[f"{v:.4f}" for v in importances.values[::-1]],
    textposition="outside",
))
fig_imp.update_layout(
    title=f"Top 10 Churn Drivers — {best_name}",
    xaxis_title="Importance",
    template="plotly_dark",
    width=800, height=420,
    margin=dict(l=0, r=80, t=50, b=0),
)
fig_imp.show()

## 12. Churn Risk Segmentation

Customers in the test set are segmented by predicted churn probability:

| Segment | Probability Range |
|---|---|
| 🟢 Low Risk | < 33% |
| 🟡 Medium Risk | 33% – 66% |
| 🔴 High Risk | > 66% |

In [ ]:
proba_s = pd.Series(probas[best_name])

low    = int((proba_s < 0.33).sum())
medium = int(((proba_s >= 0.33) & (proba_s < 0.66)).sum())
high   = int((proba_s >= 0.66).sum())
total  = len(proba_s)

print(f"🟢 Low Risk    (<33%) : {low:,}  ({low/total*100:.1f}%)")
print(f"🟡 Medium Risk (33-66%): {medium:,}  ({medium/total*100:.1f}%)")
print(f"🔴 High Risk   (>66%) : {high:,}  ({high/total*100:.1f}%)")

fig_dist = go.Figure(go.Bar(
    x=["🟢 Low Risk\n(<33%)", "🟡 Medium Risk\n(33–66%)", "🔴 High Risk\n(>66%)"],
    y=[low, medium, high],
    marker_color=["#2ecc71", "#f39c12", "#e74c3c"],
    text=[f"{low:,}", f"{medium:,}", f"{high:,}"],
    textposition="outside",
))
fig_dist.update_layout(
    title="Churn Risk Distribution (Test Set)",
    yaxis_title="Number of Customers",
    template="plotly_dark",
    showlegend=False,
    width=600, height=380,
)
fig_dist.show()

## 13. Churn Probability Distribution Histogram

In [ ]:
fig_hist = go.Figure()

for label, color, name in [(0, "#2ecc71", "Retained"), (1, "#e74c3c", "Churned")]:
    mask = y_test == label
    fig_hist.add_trace(go.Histogram(
        x=proba_s[mask.values],
        name=name,
        opacity=0.75,
        marker_color=color,
        nbinsx=40,
    ))

fig_hist.update_layout(
    title=f"Predicted Churn Probability Distribution — {best_name}",
    xaxis_title="Churn Probability",
    yaxis_title="Count",
    barmode="overlay",
    template="plotly_dark",
    width=800, height=400,
    legend=dict(orientation="h", y=-0.2),
)
fig_hist.show()

## 14. Export High-Risk Customers

In [ ]:
# Rebuild test set with original index to map back to customer IDs
X_test_full = X_test.copy()
X_test_full["churn_probability"] = probas[best_name]
X_test_full["actual_churned"]    = y_test.values
X_test_full["risk_segment"] = pd.cut(
    X_test_full["churn_probability"],
    bins=[0, 0.33, 0.66, 1.0],
    labels=["Low", "Medium", "High"]
)

high_risk_df = (
    X_test_full[X_test_full["risk_segment"] == "High"]
    .sort_values("churn_probability", ascending=False)
)

print(f"High-risk customers in test set: {len(high_risk_df):,}")
high_risk_df[["churn_probability", "total_orders", "total_spent",
              "avg_review_score", "avg_delivery_days", "actual_churned"]].head(10)

In [ ]:
# Optional: save to CSV
high_risk_df.to_csv("high_risk_customers.csv", index=False)
print("Saved → high_risk_customers.csv")

## 15. Summary

| Metric | Value |
|---|---|
| Best Model | see `best_name` |
| Churn Threshold | 180 days |
| Test Size | 20% |
| Data Source | `olist_warehouse.duckdb` |

**Key takeaways:**
- `days_since_last_order` is excluded from features (label leakage prevention)
- Class imbalance is handled via `class_weight="balanced"` in RF and LR
- GBM typically achieves the best AUC-ROC on this dataset
- High-risk customers (churn prob > 66%) are exported to `high_risk_customers.csv` for CRM targeting

---
*Data: Olist Brazilian E-Commerce — DuckDB Data Warehouse*